# Template Definition

In [2]:
from dotenv import load_dotenv
assert load_dotenv()

In [3]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_openai import ChatOpenAI 
from langchain_core.output_parsers import StrOutputParser

In [14]:
prompt_template = PromptTemplate.from_template(
"""
You will be asked by the user to compare two list of relations extracted from a text specification.
You will recieve the specification text in input and two list of relations: the predicted relations and the correct relations.

Each relation is a dictionary with two entries, representing the two classes involved in the relation and their cardinality.
For example, if class A and B are in relation with cardinalityx and cardinalityy the dictionary will be:
{{
    'A': cardinalityx,
    'B': cardinalityy
}}

You will output the intersection between these two lists, keeping in mind that similar names can be used and they are acceptable.
Include only the class from the correct list in the intersection. Use the context from the specification to understand if a class name is
acceptable.

For example, if the correct relation list is [{{'person':'1..1', 'address':'1..*'}}] and the predicted relation list is [{{'human':'1..1', 'home':'1..*'}}] the intersection should be:
[{{'person':'1..1', 'address':'1..*'}}].

Output only the intersection list without further explanation.

##############

The specification text is:

{text}

The predicted list is:

{predicted}

The correct list is:

{correct}

##############

The intersection list is:
"""
)

In [17]:
def check_relations_llm(predicted, correct, path_to_check):
    """
    Compute intersection between predicted and correct relation lists using an LLM.
    Args:
        predicted (list): List of predicted relations
        correct (list): List of correct relations
        path_to_check (str): Path to the description file to use as context
    Returns:
        dict: A dictionary with precision, recall, f1 score
    """
    try:
        model = ChatOpenAI(model="gpt-4o-mini")
        chain = prompt_template | model | StrOutputParser()

        res = chain.invoke({"text": open("/".join(path_to_check.split("/")[:-1])+'/description.md').read(),
                        "predicted": predicted,
                        "correct":correct})

        # Parse the intersection list from the LLM response
        intr = eval(res)
        print(f"LLM Intersection: {intr}")

        # Validate the intersection list contains only relations from correct
        assert set(map(str, intr)) <= set(map(str, correct))

        # Calculate metrics
        precision = len(intr) / len(predicted) if predicted else 0
        recall = len(intr) / len(correct) if correct else 0

        # Calculate F1 score
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        return {
            "precision": round(precision, 2),
            "recall": round(recall, 2),
            "f1": round(f1, 2),
            "len_": len(correct)
        }

    except Exception as e:
        print(f"Error during LLM evaluation: {e}")
        # Return zeros on error
        return {
            "precision": 0.0,
            "recall": 0.0, 
            "f1": 0.0,
            "len_": len(correct)
        }

In [18]:
# Example usage
predicted = [{"person":"1..1", "address":"1..*"}] 
correct = [{"human":"1..1", "home":"1..*"}]

check_relations_llm(predicted, correct, "../dataset/AirTravel/description.md")

LLM Intersection: []


{'precision': 0.0, 'recall': 0.0, 'f1': 0, 'len_': 1}